In [49]:
# Imports
import os
import cv2
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision import datasets
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.model_selection import train_test_split
from PIL import Image

import pandas as pd
import numpy as np

In [50]:
# Config
DATA_DIR = '../data/sample'
LABELS_CSV = os.path.join(DATA_DIR, 'sample_labels.csv')
IMAGE_DIR = os.path.join(DATA_DIR, 'images')
CHECKPOINT_DIR = '../checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

CLASS_LIST = [
    "Atelectasis",
    "Cardiomegaly",
    "Effusion",
    "Infiltration",
    "Mass",
    "Nodule",
    "Pneumonia",
    "Pneumothorax",
    "Consolidation",
    "Edema",
    "Emphysema",
    "Fibrosis",
    "Pleural_Thickening",
    "Hernia"
]
BATCH_SIZE  = 32
NUM_EPOCHS = 10
VAL_SPLIT = 0.2
IMAGE_SIZE = 224

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')


Using device: cpu


In [51]:
# Micilaneous Functions

def apply_clahe(image): 
    image = np.array(image.convert('L'))
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    image = clahe.apply(image)
    image = Image.fromarray(image).convert('RGB')
    return image

def apply_transformations(type):
    transformation_list = [transforms.Resize((IMAGE_SIZE, IMAGE_SIZE))]
    if type == 'train':
        transformation_list.append(transforms.RandomHorizontalFlip())
    transformation_list.extend([
        transforms.ToTensor(),
        transforms.Normalize(
            [0.485, 0.456, 0.406],
            [0.229, 0.224, 0.225]
        )    
    ])
    return transforms.Compose(transformation_list)

In [52]:
# Dataset Object

class ChestXRayDataset(Dataset):
    def __init__(self, dataframe, image_dir, class_list, transform=None):
        self.df = dataframe
        self.image_dir = image_dir
        self.class_list = class_list
        self.class_to_index = {
            cls_name: idx for idx, cls_name in enumerate(class_list)
        }
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row   = self.df.iloc[index]
        image = Image.open(os.path.join(self.image_dir, row['Image Index']))
        image = apply_clahe(image)
        if self.transform:
            image = self.transform(image)

        raw_labels = row['Finding Labels']
        label = torch.zeros(len(self.class_list), dtype=torch.float32)

        if pd.notna(raw_labels) and raw_labels != "No Finding":
            for cls in str(raw_labels).split('|'):
                if cls in self.class_to_index:
                    label[self.class_to_index[cls]] = 1.0

        return image, label


In [ ]:
# Dataset Loading

df = pd.read_csv(LABELS_CSV)
train_df, val_df = train_test_split(df, test_size=VAL_SPLIT, random_state=42)

train_dataset = ChestXRayDataset(
    dataframe=train_df,
    image_dir=IMAGE_DIR,
    class_list=CLASS_LIST,
    transform=apply_transformations('train')
)

val_dataset = ChestXRayDataset(
    dataframe=val_df,
    image_dir=IMAGE_DIR,
    class_list=CLASS_LIST,
    transform=apply_transformations('val')
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=5, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')


Train batches: 141 | Val batches: 36
